In [ ]:
# pgd-cifar10: normalize CWD to project root
import os
import pathlib

while (
    not pathlib.Path("pyproject.toml").exists() and pathlib.Path.cwd() != pathlib.Path.cwd().parent
):
    os.chdir("..")

In [ ]:
!pytest tests/test_attacks/ -q

# Thực nghiệm — Tổng hợp tấn công đối kháng (white-box, gray-box, black-box)

Notebook này tổng hợp toàn bộ thực nghiệm tấn công của dự án `pgd-cifar10-experiment` trên CIFAR-10. Mục tiêu là trình bày trực quan và đầy đủ:

1. **Giả định về mô hình Target/Victim** — kiến trúc và trạng thái huấn luyện được xét.
2. **Ngữ cảnh tấn công** — phân loại white-box / gray-box / black-box theo mức kiến thức của bên tấn công.
3. **Dữ liệu sử dụng** — CIFAR-10 test set ở miền pixel `[0, 1]`.
4. **Độ mạnh, hiệu quả tấn công** — đánh giá đồng thời qua *success rate*, *perturbation size*, và *số bước / thời gian*.

Tất cả số liệu ở các bảng và hình bên dưới đều được lấy lại từ kết quả thực nghiệm đã sinh ở các notebook trước (NB04, NB08, NB09) thông qua các CSV trong `results/tables/`. Notebook không huấn luyện lại model — chỉ tổng hợp, so sánh và minh họa.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.reporting.constants import ARCHES, NB04_ATTACK_NAMES, SEED
from src.reporting.io import read_csv
from src.reporting.loaders import evaluation_inputs
from src.reporting.registry import reporting_model_pair
from src.visualize.style import apply_style

apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 3.1 Giả định về mô hình Target/Victim

- **Kiến trúc**: hai họ mô hình tiêu biểu cho CNN và Transformer trên ảnh nhỏ — `resnet18` và `vit_tiny`. Cả hai được build qua `src.models.builders` và bọc bởi `Normalizer` để nhận trực tiếp ảnh raw `[0, 1]`; chuẩn hoá CIFAR-10 (mean/std) được gắn cố định vào model, do đó tấn công thực hiện trong miền pixel.
- **Trạng thái huấn luyện** (variant): mỗi kiến trúc có hai checkpoint với cùng `seed=42`:
  - `clean` — huấn luyện chuẩn (chỉ cross-entropy).
  - `adv` (`adversarial`) — adversarial training với inner attack APGD-CE-10, `epsilon=8/255`.
- **Số seed**: project đã chuyển sang single-seed (`SEED=42`) cho toàn bộ phase thực nghiệm; các bảng `mean/std` (ví dụ bảng Square) bằng `std` rỗng khi `n_runs=1`.
- **Quy ước**: trong gray-box transfer, *surrogate* là mô hình tấn công tạo perturbation, *victim* là mô hình bị tấn công thực sự.

In [ ]:
model_rows = []
for arch in ARCHES:
    pair = reporting_model_pair(arch, SEED)
    model_rows.append(
        {
            "architecture": arch,
            "seed": SEED,
            "clean_checkpoint": "available" if pair.clean.exists else "missing",
            "adv_checkpoint": "available" if pair.adversarial.exists else "missing",
            "clean_path": str(pair.clean.path),
            "adv_path": str(pair.adversarial.path),
        }
    )
display(pd.DataFrame(model_rows))

## 3.2 Ngữ cảnh tấn công (threat models)

Mỗi ngữ cảnh khác nhau ở **mức kiến thức** bên tấn công có về mô hình target và **cách truy cập** (gradient hay chỉ truy vấn).

| Ngữ cảnh | Kiến thức về target | Có gradient? | Có truy cập trọng số? | Attack tiêu biểu trong project |
|---|---|---|---|---|
| **white-box** | Đầy đủ (kiến trúc + trọng số) | Có | Có | FGSM, BIM-10, PGD-{10,40,100}, APGD-CE-{10,100} |
| **gray-box** | Biết kiến trúc nhưng trọng số khác (dùng surrogate) | Có (trên surrogate) | Trên surrogate | Gray-box transfer: surrogate `clean` → victim `adv` |
| **black-box** | Chỉ truy vấn `f(x)` (logits/label); không gradient | Không | Không | Square Attack (query-based, 5000 queries) |

White-box thường được dùng để **đặt cận trên cho mức độ thành công** của một họ tấn công gradient-based — vì kẻ tấn công biết mọi thứ. Black-box và gray-box là các threat model thực tế hơn (kẻ tấn công không có quyền truy cập trọng số) — kết quả ở những ngữ cảnh này phản ánh độ vững chắc của mô hình trong điều kiện realistic.

In [ ]:
threat_models = pd.DataFrame(
    [
        {
            "context": "white-box",
            "attacker_knowledge": "Architecture + weights (full access)",
            "uses_gradient": True,
            "query_budget": "unlimited",
            "attacks_used": ", ".join(NB04_ATTACK_NAMES),
        },
        {
            "context": "gray-box",
            "attacker_knowledge": "Same arch, different weights (surrogate)",
            "uses_gradient": True,
            "query_budget": "unlimited (on surrogate)",
            "attacks_used": "PGD-10 transfer (clean -> adv)",
        },
        {
            "context": "black-box",
            "attacker_knowledge": "Query access only (logits, no gradients)",
            "uses_gradient": False,
            "query_budget": "5000 per sample",
            "attacks_used": "Square Attack (Linf, p_init=0.05)",
        },
    ]
)
display(threat_models)

## 3.3 Dữ liệu sử dụng — CIFAR-10

- **Dataset**: CIFAR-10, 10 lớp, 50 000 ảnh train / 10 000 ảnh test, mỗi ảnh 32×32 RGB.
- **Miền pixel**: ảnh được giữ ở `[0, 1]`; chuẩn hoá channel-wise (mean/std CIFAR-10) được thực hiện *bên trong* model thông qua `Normalizer`. Nhờ vậy ràng buộc `Linf(epsilon)` cũng áp dụng trên miền pixel `[0, 1]`, tương đương với `epsilon=8/255 ≈ 0.0314`.
- **Set đánh giá tấn công**: 10 000 ảnh test (notebook chỉ load một subset nhỏ ở phần minh hoạ định tính để giữ runtime nhẹ; tất cả CSV tổng hợp đều được sinh từ full test set qua `scripts/run_white_box.py`, `scripts/run_black_box_square.py`, `scripts/run_transfer.py`).
- **Reproducibility**: dùng `SEED=42` cho tất cả RNG (Python, NumPy, PyTorch). `evaluation_loader` và `evaluation_inputs` có cùng seed, có fallback synthetic khi dataset không tải về được.

In [ ]:
CIFAR10_CLASSES = [
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

x_demo, y_demo = evaluation_inputs(8)
fig, axes = plt.subplots(2, 4, figsize=(8, 4.2))
for ax, img, lbl in zip(axes.flat, x_demo, y_demo, strict=False):
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(CIFAR10_CLASSES[int(lbl)], fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("CIFAR-10 — 8 mẫu test (raw, [0, 1])", y=1.02)
fig.tight_layout()
plt.show()

## 3.4 Phương pháp đánh giá hiệu quả tấn công

Mỗi attack được đánh giá đồng thời trên ba trục:

**Success rate**

- `asr` *(robust error)* — tỷ lệ ảnh adversarial bị mô hình victim dự đoán **sai** trên toàn bộ test set. Tương đương `1 - robust_acc`.
- `conditional_asr` — tỷ lệ tấn công thành công chỉ tính trên các mẫu mà clean model **dự đoán đúng**: `(adv_pred ≠ y)[clean_correct].mean()`. Đây là *true ASR* theo đúng định nghĩa kinh điển; nó cô lập “tấn công làm hỏng dự đoán đúng ban đầu” khỏi “mô hình vốn đã sai”.
- `robust_acc` — accuracy của victim trên ảnh đã bị attack. Càng thấp ⇒ attack càng mạnh, hoặc model càng yếu.

**Perturbation size**

- `linf_actual` / `linf_mean` — giá trị `||x_adv - x||_∞` trung bình. Với các iterative attack có random_start (PGD, APGD), `linf_actual` thường rất gần `epsilon=8/255`; với FGSM (single-step) cũng đạt budget vì sign-step.
- `epsilon_actual_ratio` — `linf_mean / epsilon`. Lớn hơn 1 nghĩa là vi phạm budget; lý tưởng ≈ 1.0 (đầy budget, hợp lệ).

**Số bước / thời gian**

- `num_steps` — số bước iterative, hoặc số queries với Square Attack.
- `time_per_image_ms` — thời gian wall-clock cho mỗi ảnh, bao gồm cả attack generation và inference của victim.

Tất cả attack đều bị verify qua `verify_perturbation()` sau mỗi batch để đảm bảo hợp lệ `Linf` và miền pixel `[0, 1]`.

## 3.5 Kết quả white-box

Sử dụng bảng tổng hợp white-box do `nb04_main_results()` sinh (`results/tables/main_results.csv`). Bảng này chạy 7 attack chuẩn (FGSM, BIM-10, PGD-{10,40,100}, APGD-CE-{10,100}) trên cả hai kiến trúc, victim là `clean` checkpoint.

In [ ]:
wb_path = Path("results/tables/main_results.csv")
wb_rows = read_csv(wb_path)
df_wb = pd.DataFrame(wb_rows) if wb_rows else None
if df_wb is not None and not df_wb.empty:
    display_cols = [
        "arch",
        "attack",
        "num_steps",
        "asr",
        "robust_acc",
        "linf_actual",
        "epsilon_actual_ratio",
        "time_per_image_ms",
    ]
    df_wb_view = df_wb[[c for c in display_cols if c in df_wb.columns]].copy()
    print(f"Loaded {len(df_wb)} rows from {wb_path}")
    display(df_wb_view)
else:
    print(f"{wb_path} chưa tồn tại — chạy scripts/run_white_box.py + nb04_main_results() để sinh.")

In [ ]:
def _cell_value(sub: pd.DataFrame, attack: str, col: str) -> float:
    if attack not in sub.index:
        return 0.0
    raw = sub.loc[attack, col]
    try:
        return float(raw) if raw not in (None, "") else 0.0
    except (TypeError, ValueError):
        return 0.0


if df_wb is not None and not df_wb.empty:
    fig, (ax_asr, ax_time) = plt.subplots(1, 2, figsize=(11, 4))
    archs_in_table = sorted(df_wb["arch"].unique())
    attack_order = [a for a in NB04_ATTACK_NAMES if a in df_wb["attack"].unique()]
    x_idx = np.arange(len(attack_order))
    width = 0.8 / max(len(archs_in_table), 1)
    for i, arch in enumerate(archs_in_table):
        sub = df_wb[df_wb["arch"] == arch].set_index("attack")
        asr_vals = [_cell_value(sub, a, "asr") for a in attack_order]
        time_vals = [_cell_value(sub, a, "time_per_image_ms") for a in attack_order]
        offset = i * width - width * (len(archs_in_table) - 1) / 2
        ax_asr.bar(x_idx + offset, asr_vals, width, label=arch)
        ax_time.bar(x_idx + offset, time_vals, width, label=arch)
    for ax, title, ylabel in (
        (ax_asr, "White-box ASR theo attack & kiến trúc", "ASR"),
        (ax_time, "Thời gian / ảnh (ms)", "time_per_image_ms"),
    ):
        ax.set_xticks(x_idx)
        ax.set_xticklabels(attack_order, rotation=30, ha="right")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("Bỏ qua biểu đồ white-box: chưa có dữ liệu.")

**Đọc bảng white-box:**

- PGD-10/40/100 và APGD-CE-100 thường đạt `asr ≈ 1.0` và `robust_acc ≈ 0` trên clean model — đúng kỳ vọng cho white-box: kẻ tấn công có gradient nên với `epsilon=8/255` gần như luôn vượt được mô hình huấn luyện chuẩn.
- FGSM (single-step) đã đủ để hạ accuracy xuống rất thấp nhưng kém hiệu quả hơn iterative attack — dùng làm baseline.
- `epsilon_actual_ratio` của các iterative attack ≈ 1.0 (đầy budget); FGSM cũng ≈ 1.0 vì sign-step. Đây là kiểm chứng tính hợp lệ của attack.
- `time_per_image_ms` tăng tuyến tính theo `num_steps`: PGD-100 ≈ 10× PGD-10, APGD-CE-100 đắt hơn nữa vì có restarts.

## 3.6 Kết quả gray-box (transfer)

Gray-box trong project: surrogate là model `clean` cùng kiến trúc, victim là model `adv` (đã adversarial training). Attack được sinh từ surrogate rồi áp dụng lên victim. Kết quả lấy từ `results/tables/09_gray_box.csv` (do `nb09_gray_box_summary()` sinh).

In [ ]:
gb_path = Path("results/tables/09_gray_box.csv")
gb_rows = read_csv(gb_path)
df_gb = pd.DataFrame(gb_rows) if gb_rows else None
if df_gb is not None and not df_gb.empty:
    print(f"Loaded {len(df_gb)} rows from {gb_path}")
    display(df_gb)
else:
    print(f"{gb_path} chưa tồn tại — chạy scripts/run_transfer.py --mode gray_box trước.")

In [ ]:
def _asr_mean_for(arch: str, variant: str) -> float:
    if df_gb is None:
        return 0.0
    sel = df_gb[(df_gb["arch"] == arch) & (df_gb["victim_variant"] == variant)]
    if sel.empty:
        return 0.0
    raw = sel["asr_mean"].iloc[0]
    try:
        return float(raw) if raw not in (None, "") else 0.0
    except (TypeError, ValueError):
        return 0.0


if df_gb is not None and not df_gb.empty:
    archs_gb = sorted(df_gb["arch"].unique())
    variants_gb = sorted(df_gb["victim_variant"].unique())
    fig, ax = plt.subplots(figsize=(max(5, 1.8 * len(archs_gb)), 3.6))
    width = 0.35
    indices = np.arange(len(archs_gb))
    for offset_idx, variant in enumerate(variants_gb):
        means = [_asr_mean_for(arch, variant) for arch in archs_gb]
        ax.bar(indices + (offset_idx - 0.5) * width, means, width, label=f"victim={variant}")
    ax.set_xticks(indices)
    ax.set_xticklabels(archs_gb)
    ax.set_ylabel("ASR (transfer, victim view)")
    ax.set_title(
        "Gray-box transfer: surrogate=clean -> victim=clean/adv\nattack=PGD-10, CIFAR-10 test"
    )
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Bỏ qua biểu đồ gray-box: chưa có dữ liệu.")

**Đọc bảng gray-box:**

- `victim=clean` ⇒ surrogate và victim có cùng phân phối nội bộ; ASR transfer rất cao, gần như white-box. Đây là sanity check cho transfer pipeline.
- `victim=adv` ⇒ adversarial training làm giảm rõ rệt khả năng transfer của perturbation: ASR thấp hơn đáng kể so với cùng attack trên victim clean. Nói cách khác, **adv training tạo decision boundary khác đủ để chống một phần transfer attack**.
- Khoảng cách giữa hai cột (`clean` vs `adv`) chính là *transfer robustness gap* — một metric đánh giá hiệu quả phòng thủ đối với threat model gray-box thực tế.

## 3.7 Kết quả black-box (Square Attack)

Black-box trong project: Square Attack, query budget 5000 truy vấn mỗi mẫu, `epsilon=8/255`, `p_init=0.05`. Bảng được sinh bởi `nb08_query_black_box_table()` từ MLflow hoặc fallback `results/logs/*.json`.

In [ ]:
bb_path = Path("results/tables/08_query_black_box.csv")
bb_rows = read_csv(bb_path)
df_bb = pd.DataFrame(bb_rows) if bb_rows else None
if df_bb is not None and not df_bb.empty:
    bb_cols = [
        "arch",
        "variant",
        "num_queries",
        "asr_mean",
        "robust_acc_mean",
        "time_per_image_ms_mean",
        "n_runs",
        "status",
    ]
    df_bb_view = df_bb[[c for c in bb_cols if c in df_bb.columns]].copy()
    print(f"Loaded {len(df_bb)} rows from {bb_path}")
    display(df_bb_view)
else:
    print(
        f"{bb_path} chưa tồn tại — chạy scripts/run_black_box_square.py + "
        f"nb08_query_black_box_table() trước."
    )

In [ ]:
if df_bb is not None and not df_bb.empty and "asr_mean" in df_bb.columns:
    df_plot = df_bb.copy()
    df_plot["asr_mean"] = pd.to_numeric(df_plot["asr_mean"], errors="coerce")
    df_plot["time_per_image_ms_mean"] = pd.to_numeric(
        df_plot["time_per_image_ms_mean"], errors="coerce"
    )
    df_plot = df_plot.dropna(subset=["asr_mean", "time_per_image_ms_mean"])
    if not df_plot.empty:
        fig, ax = plt.subplots(figsize=(6, 4))
        for variant, marker in (("clean", "o"), ("adv", "x")):
            sub = df_plot[df_plot["variant"] == variant]
            ax.scatter(
                sub["time_per_image_ms_mean"],
                sub["asr_mean"],
                marker=marker,
                label=f"victim={variant}",
            )
            for _, row in sub.iterrows():
                ax.annotate(
                    str(row["arch"]),
                    (row["time_per_image_ms_mean"], row["asr_mean"]),
                    fontsize=7,
                )
        ax.set_xlabel("time per image (ms)")
        ax.set_ylabel("ASR")
        ax.set_title("Black-box (Square): chi phí truy vấn vs hiệu quả")
        ax.legend()
        fig.tight_layout()
        plt.show()
    else:
        print("Bỏ qua biểu đồ Square: thiếu cột asr_mean hoặc time_per_image_ms_mean.")
else:
    print("Bỏ qua biểu đồ Square: chưa có dữ liệu.")

**Đọc bảng black-box:**

- Square Attack vẫn đạt ASR đáng kể trên victim `clean` dù không có gradient — chứng tỏ với 5000 queries/mẫu mô hình clean không đủ vững.
- Trên victim `adv`, ASR thấp hơn rõ rệt — adversarial training cũng có hiệu quả phòng thủ phần nào với black-box query attack, dù không phải mục tiêu trực tiếp của adv training.
- `time_per_image_ms_mean` của Square cao hơn các white-box attack vì cần đánh giá toàn bộ 5000 queries; đổi lại không cần backward pass.

## 3.8 So sánh trực tiếp ba ngữ cảnh tấn công

Để có cái nhìn hợp nhất, ta đặt cạnh nhau ASR của ba threat model trên cùng kiến trúc:
- White-box: ASR của PGD-10 trên victim `clean` (lấy từ `main_results.csv`).
- Gray-box: ASR transfer của PGD-10 trên victim `adv` (lấy từ `09_gray_box.csv`).
- Black-box: ASR của Square 5000 queries trên victim `clean` và `adv` (lấy từ `08_query_black_box.csv`).

Lưu ý các con số *không hoàn toàn so sánh trực tiếp* được vì victim khác variant; mục đích là cho thấy **mức độ giảm hiệu quả khi kẻ tấn công mất quyền truy cập gradient/trọng số**.

In [ ]:
def _safe_float(value, default=float("nan")):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


compare_rows = []
for arch in ARCHES:
    if df_wb is not None:
        sub = df_wb[(df_wb["arch"] == arch) & (df_wb["attack"] == "pgd_10")]
        if not sub.empty:
            compare_rows.append(
                {
                    "arch": arch,
                    "context": "white-box",
                    "attack": "PGD-10",
                    "victim": "clean",
                    "asr": _safe_float(sub.iloc[0].get("asr")),
                    "time_ms": _safe_float(sub.iloc[0].get("time_per_image_ms")),
                }
            )
    if df_gb is not None:
        sub = df_gb[(df_gb["arch"] == arch) & (df_gb["victim_variant"] == "adv")]
        if not sub.empty:
            compare_rows.append(
                {
                    "arch": arch,
                    "context": "gray-box",
                    "attack": "PGD-10 transfer",
                    "victim": "adv",
                    "asr": _safe_float(sub.iloc[0].get("asr_mean")),
                    "time_ms": float("nan"),
                }
            )
    if df_bb is not None:
        for variant in ("clean", "adv"):
            sub = df_bb[(df_bb["arch"] == arch) & (df_bb["variant"] == variant)]
            if not sub.empty:
                compare_rows.append(
                    {
                        "arch": arch,
                        "context": "black-box",
                        "attack": "Square 5000",
                        "victim": variant,
                        "asr": _safe_float(sub.iloc[0].get("asr_mean")),
                        "time_ms": _safe_float(sub.iloc[0].get("time_per_image_ms_mean")),
                    }
                )
df_compare = pd.DataFrame(compare_rows)
display(df_compare)

In [ ]:
if not df_compare.empty:
    pivot = df_compare.pivot_table(
        index=["arch", "context"], columns="victim", values="asr", aggfunc="mean"
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel("ASR")
    ax.set_title("ASR theo ngữ cảnh × kiến trúc × variant victim")
    ax.set_xticklabels(
        [" / ".join(map(str, idx)) for idx in pivot.index],
        rotation=30,
        ha="right",
    )
    ax.legend(title="victim variant")
    fig.tight_layout()
    plt.show()
else:
    print("Bỏ qua biểu đồ so sánh: chưa có dữ liệu của bất kỳ ngữ cảnh nào.")

## 3.9 Minh hoạ định tính: perturbation và phân biệt mắt thường

Phần này dùng `make_perturbation_panel()` để dựng panel 3 cột: clean / adversarial / `10× perturbation`. Nếu checkpoint `resnet18 / clean / seed=42` không có sẵn (clone sạch), ta dùng model khởi tạo ngẫu nhiên để giữ notebook chạy được — kết quả không có ý nghĩa định lượng nhưng vẫn cho thấy cấu trúc perturbation của PGD-10.

In [ ]:
from src.reporting.attack import build_attack_for_report
from src.reporting.registry import build_fresh_model
from src.utils.seed import set_all_seeds
from src.visualize.perturbation_panels import make_perturbation_panel

set_all_seeds(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pair = reporting_model_pair("resnet18", SEED)
victim = pair.clean.load_or_none()
have_real = victim is not None
if victim is None:
    victim = build_fresh_model("resnet18")
victim = victim.to(device).eval()

x_q, y_q = evaluation_inputs(8)
x_q = x_q.to(device)
y_q = y_q.to(device)

attack = build_attack_for_report("pgd_10")
x_adv = attack.perturb(victim, x_q, y_q)

with torch.no_grad():
    pred_clean = victim(x_q).argmax(dim=1).cpu()
    pred_adv = victim(x_adv).argmax(dim=1).cpu()
    y_cpu = y_q.cpu()

delta = (x_adv - x_q).abs()
linf_per_sample = delta.flatten(1).max(dim=1).values
print(f"Linf trung bình (8 mẫu): {linf_per_sample.mean().item():.5f}")
print(f"Linf max  (8 mẫu): {linf_per_sample.max().item():.5f}")
print(f"epsilon ràng buộc : {attack.config.epsilon:.5f}")
print(f"clean predictions : {pred_clean.tolist()}")
print(f"adv   predictions : {pred_adv.tolist()}")
print(f"true labels       : {y_cpu.tolist()}")
if not have_real:
    print(
        "\nLưu ý: checkpoint clean ResNet-18 chưa có; minh hoạ chạy trên trọng số khởi tạo "
        "ngẫu nhiên — chỉ để xem cấu trúc perturbation."
    )

In [ ]:
ckpt_state = "real clean ckpt" if have_real else "random weights — smoke only"
title_real = f"PGD-10 trên ResNet-18 ({ckpt_state}), epsilon=8/255, alpha=2/255, seed={SEED}"
fig = make_perturbation_panel(x_q.detach().cpu(), x_adv.detach().cpu(), title_real)
plt.show()

**Đọc panel perturbation:**

- Cột giữa (adversarial) gần như không phân biệt được với cột trái (clean) bằng mắt thường — đặc trưng cốt lõi của Linf attack ở `epsilon=8/255` trên CIFAR-10.
- Cột phải (`10× perturbation`) khuếch đại độ lệch để dễ nhìn cấu trúc; với PGD ta thường thấy nhiễu tần số cao (high-frequency texture), phân bố trên toàn ảnh chứ không tập trung tại đối tượng.
- Khi có real checkpoint, các dự đoán `adv predictions` thường khác `true labels` — phản ánh attack thành công. Với random weights, dự đoán clean lẫn adv đều không khớp label thật → không có ý nghĩa định lượng.

## 3.10 Tổng kết

Thực nghiệm 3 trong dự án này có những kết luận chính:

1. **Cận trên white-box**: trên CIFAR-10 với `epsilon=8/255`, ResNet-18 / ViT-Tiny huấn luyện chuẩn gần như luôn bị PGD/APGD đánh thủng (ASR ≈ 1.0). FGSM single-step đã đủ để gây thiệt hại lớn — phục vụ làm baseline về tốc độ.
2. **Adversarial training là phòng thủ chính**: chỉ adversarial training (APGD-CE-10 inner) mới giảm được robust error ở mức ý nghĩa. Trên gray-box (transfer PGD-10), ASR victim `adv` thấp hơn rõ rệt so với victim `clean`; trên black-box (Square 5000) cũng có pattern tương tự.
3. **Trade-off thời gian / hiệu quả**: PGD-10 đã chiếm phần lớn ASR đạt được bởi PGD-100; PGD-100 và APGD-CE-100 chỉ đem lại lợi ích nhỏ về ASR nhưng tăng chi phí ≥ 10×. Square Attack có chi phí cao nhưng không cần gradient — phù hợp với threat model thực tế khi kẻ tấn công chỉ có API truy vấn.
4. **Tính hợp lệ của attack**: `linf_actual / epsilon ≈ 1.0` đối với mọi iterative attack — kiểm chứng rằng attack thực sự đầy budget và không vi phạm ràng buộc; `verify_perturbation` được gọi sau mỗi batch để fail fast khi có vi phạm.

**Tài liệu tham khảo trong project**: NB04 (white-box), NB08 (defense + black-box), NB09 (transfer gray-box), NB10 (so sánh kiến trúc), NB11 (limitations).